# 06 — Push Model to HuggingFace Hub
## LLM Lie Detector | Hallucination Detection Pipeline

This notebook pushes the fine-tuned LoRA adapter to HuggingFace Hub so it is 
publicly accessible to the NLP community.

### Goals
- Load the best checkpoint (checkpoint-2688)
- Push the LoRA adapter to HuggingFace Hub
- Write a model card describing the model, training, and results

In [1]:
import os
os.environ["PYTHONUTF8"] = "1"

from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv('../.env')
login(token=os.getenv("HF_TOKEN"))
print("Logged in successfully.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in successfully.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model_id = "meta-llama/Llama-3.2-3B-Instruct"
checkpoint_path = "../outputs/checkpoints/checkpoint-2688"

# Your HuggingFace username
hf_username = "tamimmirza"
repo_name = "llama-3.2-3b-hallucination-detector"
hub_model_id = f"{hf_username}/{repo_name}"

print(f"Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

print(f"Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto"
)

print(f"Loading LoRA adapter from {checkpoint_path}...")
model = PeftModel.from_pretrained(base_model, checkpoint_path)

print("All loaded. Ready to push.")

Loading tokenizer...
Loading base model...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loading LoRA adapter from ../outputs/checkpoints/checkpoint-2688...


W0429 13:57:31.772000 22608 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


All loaded. Ready to push.


In [3]:
print(f"Pushing model to {hub_model_id}...")
model.push_to_hub(hub_model_id, private=False)
tokenizer.push_to_hub(hub_model_id, private=False)
print(f"Done. Model live at: https://huggingface.co/{hub_model_id}")

Pushing model to darthacnologia/llama-3.2-3b-hallucination-detector...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

c:\ML Projects\llm-lie-detector\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Tamim\.cache\huggingface\hub\models--darthacnologia--llama-3.2-3b-hallucination-detector. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Done. Model live at: https://huggingface.co/darthacnologia/llama-3.2-3b-hallucination-detector


In [4]:
from huggingface_hub import ModelCard

card_content = """
---
language:
- en
license: llama3.2
base_model: meta-llama/Llama-3.2-3B-Instruct
tags:
- hallucination-detection
- text-classification
- lora
- peft
- nlp
datasets:
- truthful_qa
- pminervini/HaluEval
metrics:
- f1
- accuracy
---

# LLM Lie Detector — Hallucination Detection Model

A fine-tuned Llama 3.2 3B Instruct model for detecting hallucinations in LLM-generated answers.

## Model Description

This model is fine-tuned using LoRA (Low-Rank Adaptation) on a combined dataset of TruthfulQA 
and HaluEval to classify whether a given question-answer pair contains a hallucination.

Given a question and an answer, the model outputs either `TRUTHFUL` or `HALLUCINATED`.

## Training

- **Base model:** meta-llama/Llama-3.2-3B-Instruct
- **Method:** LoRA fine-tuning (0.14% of parameters trained)
- **Dataset:** TruthfulQA + HaluEval (15,918 labeled pairs)
- **Hardware:** NVIDIA RTX 4080 Laptop GPU (12GB VRAM)
- **Training time:** ~54 minutes

## Results

| Metric    | Baseline | This Model | Improvement |
|-----------|----------|------------|-------------|
| F1 Score  | 0.3595   | 0.9032     | +0.5438     |
| Precision | 0.2738   | 0.9078     | +0.6340     |
| Recall    | 0.5232   | 0.9033     | +0.3800     |
| Accuracy  | —        | 90.33%     | —           |

## Usage

```python
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

model_id = "meta-llama/Llama-3.2-3B-Instruct"
adapter_id = "darthacnologia/llama-3.2-3b-hallucination-detector"

tokenizer = AutoTokenizer.from_pretrained(model_id)
base_model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.bfloat16)
model = PeftModel.from_pretrained(base_model, adapter_id)

prompt = "Question: Where did fortune cookies originate?\\nAnswer: Fortune cookies originated in China.\\nVerdict:"
inputs = tokenizer(prompt, return_tensors="pt")
output = model.generate(**inputs, max_new_tokens=10)
print(tokenizer.decode(output[0], skip_special_tokens=True))
```

## Limitations

- Performs best on nuanced factual questions similar to TruthfulQA and HaluEval
- May be inconsistent on very simple common knowledge questions
- English only
- Confidence scores are binary, not continuous probabilities

## Project

Part of the LLM Lie Detector project: https://github.com/tamimmirza/llm-lie-detector
"""

card = ModelCard(card_content)
card.push_to_hub("darthacnologia/llama-3.2-3b-hallucination-detector")
print("Model card pushed successfully.")

Model card pushed successfully.
